# FewShotPainAdaptation LOSO Hyperparameter Search on Google Colab

This notebook runs a persistent Optuna search over the highest-impact learning hyperparameters for the full LOSO pipeline. Each trial evaluates the selected configuration over `N_LOSO_STEPS` held-out subjects and maximizes held-out accuracy.

Artifacts are written to Google Drive after every trial: the Optuna SQLite study, full trial payloads, trial hyperparameters, a CSV/JSON trial history, and the current best hyperparameters.


In [ ]:
!pip -q install -U pip
!pip -q install tensorflow==2.18.1 cloudpickle matplotlib numpy pandas scikit-learn scipy seaborn pydantic optuna


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/hhihn/FewShotPainAdaptation.git"
PROJECT_DIR = Path("/content/FewShotPainAdaptation")

if not PROJECT_DIR.exists():
    !git clone $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull

%cd $PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
from argparse import Namespace
from datetime import datetime, timezone
import gc
import json
import logging
from pathlib import Path
import random
import time

import numpy as np
import optuna
import pandas as pd
import tensorflow as tf
from tensorflow.keras import mixed_precision

from tests.full_loso_trial import run_full_loso_trial
from utils.logger import setup_logger

# Colab search runs are GPU-oriented. Disable this check only for syntax/debug work.
REQUIRE_GPU = True
gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs:", gpus)
if REQUIRE_GPU and not gpus:
    raise RuntimeError("No GPU detected. In Colab, use Runtime > Change runtime type > GPU.")

mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

ENABLE_DETERMINISM = False
SEED = 42
if ENABLE_DETERMINISM:
    try:
        tf.config.experimental.enable_op_determinism()
        print("Enabled deterministic TensorFlow ops")
    except Exception as exc:
        print("Deterministic ops not available:", exc)
else:
    print("Deterministic ops disabled for throughput")

tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/PainData")
RUN_ROOT = Path("/content/drive/MyDrive/FewShotPainAdaptationHparamSearches")
DATASET_SOURCE = "biovid_part_a"

# Search budget. Increase N_TRIALS and N_LOSO_STEPS for the final search.
N_TRIALS = 16
N_LOSO_STEPS = 5
LOSO_START_INDEX = 1
LOSO_STOP_INDEX = LOSO_START_INDEX + N_LOSO_STEPS - 1
TIMEOUT_SECONDS = None

# Full LOSO reports both zero-shot and k-shot held-out accuracy. The default
# objective is k-shot held-out accuracy after adaptation.
HELDOUT_METRIC = "k_shot_accuracy"  # alternatives: "zero_shot_accuracy"
STUDY_NAME = f"loso_hparam_{DATASET_SOURCE}_{HELDOUT_METRIC}"

# To resume a previous Drive-backed search after a Colab disconnect, set this to
# that run directory before executing this cell. Leave as None for a fresh run.
RESUME_RUN_DIR = None
# RESUME_RUN_DIR = Path("/content/drive/MyDrive/FewShotPainAdaptationHparamSearches/...")

if DATASET_SOURCE == "biovid_part_a":
    required_dirs = [
        DATA_DIR / "BioVid" / "PartA" / "Train",
        DATA_DIR / "BioVid" / "PartA" / "Test",
    ]
    missing = [str(path) for path in required_dirs if not path.exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing BioVid PartA directories under DATA_DIR={DATA_DIR}: {missing}"
        )
else:
    required = ["X_pre.npy", "y_heater.npy", "subjects.npy"]
    missing = [name for name in required if not (DATA_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing files in DATA_DIR={DATA_DIR}: {missing}")

if RESUME_RUN_DIR is None:
    run_stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    RUN_DIR = RUN_ROOT / f"{STUDY_NAME}-{run_stamp}"
else:
    RUN_DIR = Path(RESUME_RUN_DIR)
TRIAL_ROOT = RUN_DIR / "trials"
STUDY_DB = RUN_DIR / "study.sqlite3"
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRIAL_ROOT.mkdir(parents=True, exist_ok=True)

logger = setup_logger("colab_loso_hparam_search", level=logging.INFO)
file_handler = logging.FileHandler(RUN_DIR / "hparam_search.log")
file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s:%(lineno)d | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
)
logger.addHandler(file_handler)
logger.info("Search run directory: %s", RUN_DIR)
logger.info("Data directory: %s", DATA_DIR)
logger.info("Study database: %s", STUDY_DB)

BASE_DEFAULTS = {
    "dataset_source": DATASET_SOURCE,
    "data_variant": "real",
    "seed": SEED,
    "k_shot": 10,
    "q_query": 10,
    "task_class_ids": "0,4",
    "task_construction_mode": "single_subject",
    "fusion_method": "mean",
    "classifier_mode": "soft_knn",
    "normalize_mode": "split",
    "learning_rate": 3e-4,
    "embedding_dim": 64,
    "filters": "16,16,32,32,64,64,128",
    "tcn_attention_key_dim": 32,
    "tcn_attention_pool_size": 0,
    "use_attention": False,
    "gaussian_noise_std": 0.0,
    "supcon_loss_weight": 0.0,
    "supcon_temperature": 0.05,
    "triplet_loss_weight": 1.0,
    "triplet_margin": 0.2,
    "deterministic_ops": ENABLE_DETERMINISM,
    "num_epochs": 1,
    "tasks_per_epoch": 15000,
    "task_batch_size": 256,
    "val_tasks": 50,
    "heldout_eval_tasks": 500,
    "subject_eval_tasks": None,
    "k_shot_adaptation_steps": 10,
    "train_log_every": 25,
    "eval_log_every": 10,
    "val_batch_size": 32,
    "val_every_n_train_steps": 50,
    "summary_every_n_train_steps": 100,
    "train_prefetch_batches": 2,
    "train_progress_write_every_n_batches": 10,
    "csv_flush_every_events": 100,
    "disable_window_shift": False,
    "logging_verbosity": 1,
    "max_folds": None,
    "loso_start_index": LOSO_START_INDEX,
    "loso_stop_index": LOSO_STOP_INDEX,
}


In [ ]:
def _utc_now_iso() -> str:
    return datetime.now(tz=timezone.utc).isoformat()


def _json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, Namespace):
        return {key: _json_safe(val) for key, val in vars(value).items()}
    if isinstance(value, dict):
        return {str(key): _json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(_json_safe(payload), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


SEARCH_SPACE = {
    "learning_rate": {
        "type": "float_log",
        "range": [8e-5, 8e-4],
        "baseline": BASE_DEFAULTS["learning_rate"],
        "reason": "Highest-impact optimizer knob; range is centered around main.ipynb's 3e-4.",
    },
    "task_batch_size": {
        "type": "categorical",
        "choices": [128, 192, 256],
        "baseline": BASE_DEFAULTS["task_batch_size"],
        "reason": "Controls gradient noise and GPU memory use in episodic training.",
    },
    "tasks_per_epoch": {
        "type": "categorical",
        "choices": [7500, 10000, 15000],
        "baseline": BASE_DEFAULTS["tasks_per_epoch"],
        "reason": "Training budget per fold; includes the retained main.ipynb value.",
    },
    "normalize_mode": {
        "type": "categorical",
        "choices": ["split", "subject", "support"],
        "baseline": BASE_DEFAULTS["normalize_mode"],
        "reason": "Normalization has repeatedly mattered for subject transfer.",
    },
    "classifier_mode": {
        "type": "categorical",
        "choices": ["soft_knn", "prototype"],
        "baseline": BASE_DEFAULTS["classifier_mode"],
        "reason": "Changes how support information is used at inference.",
    },
    "fusion_method": {
        "type": "categorical",
        "choices": ["mean", "gated"],
        "baseline": BASE_DEFAULTS["fusion_method"],
        "reason": "Tests whether learned modality gates improve over stable mean fusion.",
    },
    "embedding_dim": {
        "type": "categorical",
        "choices": [48, 64, 96],
        "baseline": BASE_DEFAULTS["embedding_dim"],
        "reason": "Capacity/bias tradeoff near the retained 64-dimensional embedding.",
    },
    "filters": {
        "type": "categorical",
        "choices": [
            "16,16,32,32,64,64,128",
            "16,32,32,64,64,128",
            "16,16,32,64,64,128",
        ],
        "baseline": BASE_DEFAULTS["filters"],
        "reason": "Small architecture variants around the compact retained CNN stack.",
    },
    "gaussian_noise_std": {
        "type": "float",
        "range": [0.0, 0.03],
        "baseline": BASE_DEFAULTS["gaussian_noise_std"],
        "reason": "Light augmentation may improve held-out robustness without corrupting signals.",
    },
    "triplet_loss_weight": {
        "type": "float",
        "range": [0.5, 1.5],
        "baseline": BASE_DEFAULTS["triplet_loss_weight"],
        "reason": "Metric-learning pressure is central to the current learning objective.",
    },
    "triplet_margin": {
        "type": "categorical",
        "choices": [0.1, 0.2, 0.3],
        "baseline": BASE_DEFAULTS["triplet_margin"],
        "reason": "Controls the required class separation for triplet loss.",
    },
    "supcon_loss_weight": {
        "type": "categorical",
        "choices": [0.0, 0.05, 0.1, 0.2],
        "baseline": BASE_DEFAULTS["supcon_loss_weight"],
        "reason": "Small supervised contrastive weight may improve cohesion; large values are avoided.",
    },
    "k_shot_adaptation_steps": {
        "type": "categorical",
        "choices": [0, 5, 10, 20],
        "baseline": BASE_DEFAULTS["k_shot_adaptation_steps"],
        "reason": "Directly affects the k-shot held-out metric used as the default objective.",
    },
    "use_attention": {
        "type": "categorical",
        "choices": [False, True],
        "baseline": BASE_DEFAULTS["use_attention"],
        "reason": "Tests the optional per-modality attention path without making it mandatory.",
    },
    "tcn_attention_key_dim": {
        "type": "conditional_categorical",
        "choices_when_attention": [16, 32],
        "fixed_when_no_attention": BASE_DEFAULTS["tcn_attention_key_dim"],
        "reason": "Keeps the attention projection small if attention is enabled.",
    },
    "tcn_attention_pool_size": {
        "type": "conditional_categorical",
        "choices_when_attention": [0, 2, 4],
        "fixed_when_no_attention": BASE_DEFAULTS["tcn_attention_pool_size"],
        "reason": "Optional temporal key/value downsampling for the attention path.",
    },
}


def _sample_trial_params(trial) -> dict:
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 8e-5, 8e-4, log=True),
        "task_batch_size": trial.suggest_categorical("task_batch_size", [128, 192, 256]),
        "tasks_per_epoch": trial.suggest_categorical("tasks_per_epoch", [7500, 10000, 15000]),
        "normalize_mode": trial.suggest_categorical("normalize_mode", ["split", "subject", "support"]),
        "classifier_mode": trial.suggest_categorical("classifier_mode", ["soft_knn", "prototype"]),
        "fusion_method": trial.suggest_categorical("fusion_method", ["mean", "gated"]),
        "embedding_dim": trial.suggest_categorical("embedding_dim", [48, 64, 96]),
        "filters": trial.suggest_categorical(
            "filters",
            [
                "16,16,32,32,64,64,128",
                "16,32,32,64,64,128",
                "16,16,32,64,64,128",
            ],
        ),
        "gaussian_noise_std": trial.suggest_float("gaussian_noise_std", 0.0, 0.03),
        "triplet_loss_weight": trial.suggest_float("triplet_loss_weight", 0.5, 1.5),
        "triplet_margin": trial.suggest_categorical("triplet_margin", [0.1, 0.2, 0.3]),
        "supcon_loss_weight": trial.suggest_categorical("supcon_loss_weight", [0.0, 0.05, 0.1, 0.2]),
        "k_shot_adaptation_steps": trial.suggest_categorical("k_shot_adaptation_steps", [0, 5, 10, 20]),
        "use_attention": trial.suggest_categorical("use_attention", [False, True]),
    }
    if params["use_attention"]:
        params["tcn_attention_key_dim"] = trial.suggest_categorical(
            "tcn_attention_key_dim", [16, 32]
        )
        params["tcn_attention_pool_size"] = trial.suggest_categorical(
            "tcn_attention_pool_size", [0, 2, 4]
        )
    else:
        params["tcn_attention_key_dim"] = BASE_DEFAULTS["tcn_attention_key_dim"]
        params["tcn_attention_pool_size"] = BASE_DEFAULTS["tcn_attention_pool_size"]
    return params


def _build_trial_args(params: dict, trial_dir: Path) -> Namespace:
    values = dict(BASE_DEFAULTS)
    values.update(params)
    training_dir = trial_dir / "training_progress"
    training_dir.mkdir(parents=True, exist_ok=True)
    return Namespace(
        data_dir=str(DATA_DIR),
        dataset_source=values["dataset_source"],
        data_variant=values["data_variant"],
        seed=values["seed"],
        k_shot=values["k_shot"],
        q_query=values["q_query"],
        task_class_ids=values["task_class_ids"],
        task_construction_mode=values["task_construction_mode"],
        fusion_method=values["fusion_method"],
        classifier_mode=values["classifier_mode"],
        normalize_mode=values["normalize_mode"],
        learning_rate=values["learning_rate"],
        embedding_dim=values["embedding_dim"],
        filters=values["filters"],
        tcn_attention_key_dim=values["tcn_attention_key_dim"],
        tcn_attention_pool_size=values["tcn_attention_pool_size"],
        use_attention=values["use_attention"],
        gaussian_noise_std=values["gaussian_noise_std"],
        supcon_loss_weight=values["supcon_loss_weight"],
        supcon_temperature=values["supcon_temperature"],
        triplet_loss_weight=values["triplet_loss_weight"],
        triplet_margin=values["triplet_margin"],
        deterministic_ops=values["deterministic_ops"],
        num_epochs=values["num_epochs"],
        tasks_per_epoch=values["tasks_per_epoch"],
        task_batch_size=values["task_batch_size"],
        val_tasks=values["val_tasks"],
        heldout_eval_tasks=values["heldout_eval_tasks"],
        subject_eval_tasks=values["subject_eval_tasks"],
        k_shot_adaptation_steps=values["k_shot_adaptation_steps"],
        train_log_every=values["train_log_every"],
        eval_log_every=values["eval_log_every"],
        val_batch_size=values["val_batch_size"],
        val_every_n_train_steps=values["val_every_n_train_steps"],
        summary_every_n_train_steps=values["summary_every_n_train_steps"],
        train_prefetch_batches=values["train_prefetch_batches"],
        train_progress_write_every_n_batches=values[
            "train_progress_write_every_n_batches"
        ],
        csv_flush_every_events=values["csv_flush_every_events"],
        disable_window_shift=values["disable_window_shift"],
        logging_verbosity=values["logging_verbosity"],
        training_progress_output_dir=str(training_dir),
        model_architecture_output=str(trial_dir / "model_summary.txt"),
        skip_model_architecture_save=False,
        max_folds=values["max_folds"],
        loso_start_index=values["loso_start_index"],
        loso_stop_index=values["loso_stop_index"],
        output_json=str(trial_dir / "full_loso_results.json"),
    )


def _metric_from_result(result: dict) -> float:
    if HELDOUT_METRIC not in result["summary"]:
        raise KeyError(
            f"Unknown HELDOUT_METRIC={HELDOUT_METRIC!r}; available: "
            f"{sorted(result['summary'].keys())}"
        )
    return float(result["summary"][HELDOUT_METRIC]["mean"])


In [ ]:
def _trial_to_record(trial) -> dict:
    duration_seconds = (
        float(trial.duration.total_seconds()) if trial.duration is not None else None
    )
    return {
        "trial_number": int(trial.number),
        "state": trial.state.name,
        "value": float(trial.value) if trial.value is not None else None,
        "params": dict(trial.params),
        "user_attrs": dict(trial.user_attrs),
        "datetime_start": (
            trial.datetime_start.isoformat()
            if trial.datetime_start is not None
            else None
        ),
        "datetime_complete": (
            trial.datetime_complete.isoformat()
            if trial.datetime_complete is not None
            else None
        ),
        "duration_seconds": duration_seconds,
    }


def _persist_study(study) -> None:
    records = [_trial_to_record(trial) for trial in study.trials]
    history_payload = {
        "updated_at_utc": _utc_now_iso(),
        "study_name": study.study_name,
        "direction": "maximize",
        "objective": HELDOUT_METRIC,
        "n_loso_steps": N_LOSO_STEPS,
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "trials": records,
    }
    _write_json(RUN_DIR / "study_history.json", history_payload)

    rows = []
    for record in records:
        row = {
            "trial_number": record["trial_number"],
            "state": record["state"],
            "value": record["value"],
            "duration_seconds": record["duration_seconds"],
            "datetime_start": record["datetime_start"],
            "datetime_complete": record["datetime_complete"],
        }
        row.update({f"param_{key}": val for key, val in record["params"].items()})
        row.update({f"attr_{key}": val for key, val in record["user_attrs"].items()})
        rows.append(row)
    pd.DataFrame(rows).to_csv(RUN_DIR / "trials.csv", index=False)

    complete_trials = [
        trial
        for trial in study.trials
        if trial.state.name == "COMPLETE" and trial.value is not None
    ]
    if not complete_trials:
        return

    best_trial = max(complete_trials, key=lambda trial: float(trial.value))
    best_payload = {
        "updated_at_utc": _utc_now_iso(),
        "study_name": study.study_name,
        "objective": HELDOUT_METRIC,
        "direction": "maximize",
        "n_loso_steps": N_LOSO_STEPS,
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "best_trial_number": int(best_trial.number),
        "best_value": float(best_trial.value),
        "best_params": dict(best_trial.params),
        "best_user_attrs": dict(best_trial.user_attrs),
    }
    _write_json(RUN_DIR / "best_hyperparameters.json", best_payload)


search_config = {
    "created_at_utc": _utc_now_iso(),
    "study_name": STUDY_NAME,
    "objective": HELDOUT_METRIC,
    "direction": "maximize",
    "n_trials": N_TRIALS,
    "timeout_seconds": TIMEOUT_SECONDS,
    "n_loso_steps": N_LOSO_STEPS,
    "loso_start_index": LOSO_START_INDEX,
    "loso_stop_index": LOSO_STOP_INDEX,
    "data_dir": DATA_DIR,
    "run_dir": RUN_DIR,
    "study_db": STUDY_DB,
    "base_defaults": BASE_DEFAULTS,
    "search_space": SEARCH_SPACE,
}
_write_json(RUN_DIR / "search_config.json", search_config)

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=min(8, max(1, N_TRIALS // 2)),
    multivariate=True,
    group=True,
)
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
    direction="maximize",
    sampler=sampler,
)

BASELINE_TRIAL = {
    key: BASE_DEFAULTS[key]
    for key in [
        "learning_rate",
        "task_batch_size",
        "tasks_per_epoch",
        "normalize_mode",
        "classifier_mode",
        "fusion_method",
        "embedding_dim",
        "filters",
        "gaussian_noise_std",
        "triplet_loss_weight",
        "triplet_margin",
        "supcon_loss_weight",
        "k_shot_adaptation_steps",
        "use_attention",
    ]
}
STRONG_REGULARIZED_TRIAL = dict(BASELINE_TRIAL)
STRONG_REGULARIZED_TRIAL.update(
    {
        "fusion_method": "gated",
        "gaussian_noise_std": 0.01,
        "supcon_loss_weight": 0.05,
        "triplet_loss_weight": 0.8,
    }
)
if len(study.trials) == 0:
    study.enqueue_trial(BASELINE_TRIAL)
    study.enqueue_trial(STRONG_REGULARIZED_TRIAL)
    logger.info("Enqueued baseline and lightly regularized initial trials.")

print("Run directory:", RUN_DIR)
print("Study database:", STUDY_DB)
print("Objective:", HELDOUT_METRIC)
print("LOSO fold range:", LOSO_START_INDEX, "to", LOSO_STOP_INDEX)


In [ ]:
def objective(trial) -> float:
    tf.keras.backend.clear_session()
    gc.collect()

    params = _sample_trial_params(trial)
    trial_dir = TRIAL_ROOT / f"trial_{trial.number:04d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    args = _build_trial_args(params=params, trial_dir=trial_dir)

    hyperparameter_payload = {
        "created_at_utc": _utc_now_iso(),
        "trial_number": int(trial.number),
        "objective": HELDOUT_METRIC,
        "n_loso_steps": N_LOSO_STEPS,
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "params": params,
        "args": args,
        "base_defaults": BASE_DEFAULTS,
    }
    _write_json(trial_dir / "training_progress" / "trial_hyperparameters.json", hyperparameter_payload)

    logger.info("[Trial %s] Starting with params=%s", trial.number, params)
    started = time.perf_counter()
    try:
        result = run_full_loso_trial(args)
        score = _metric_from_result(result)
        zero_shot_mean = float(result["summary"]["zero_shot_accuracy"]["mean"])
        k_shot_mean = float(result["summary"]["k_shot_accuracy"]["mean"])
        elapsed_seconds = float(time.perf_counter() - started)

        trial.set_user_attr("trial_dir", str(trial_dir))
        trial.set_user_attr("output_json", str(args.output_json))
        trial.set_user_attr("zero_shot_accuracy_mean", zero_shot_mean)
        trial.set_user_attr("k_shot_accuracy_mean", k_shot_mean)
        trial.set_user_attr("elapsed_seconds", elapsed_seconds)
        trial.set_user_attr("num_folds", int(result["summary"]["num_folds"]))

        trial_summary = {
            "completed_at_utc": _utc_now_iso(),
            "trial_number": int(trial.number),
            "objective": HELDOUT_METRIC,
            "score": score,
            "zero_shot_accuracy_mean": zero_shot_mean,
            "k_shot_accuracy_mean": k_shot_mean,
            "elapsed_seconds": elapsed_seconds,
            "params": params,
            "args": args,
            "result_summary": result["summary"],
            "result_config": result["config"],
        }
        _write_json(trial_dir / "trial_summary.json", trial_summary)
        logger.info(
            "[Trial %s] Complete: %s=%.4f zero_shot=%.4f k_shot=%.4f elapsed=%.1fs",
            trial.number,
            HELDOUT_METRIC,
            score,
            zero_shot_mean,
            k_shot_mean,
            elapsed_seconds,
        )
        return score
    except Exception as exc:
        failure_payload = {
            "failed_at_utc": _utc_now_iso(),
            "trial_number": int(trial.number),
            "params": params,
            "args": args,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }
        _write_json(trial_dir / "trial_failure.json", failure_payload)
        logger.exception("[Trial %s] Failed", trial.number)
        raise
    finally:
        tf.keras.backend.clear_session()
        gc.collect()


def after_trial(study_obj, trial_obj) -> None:
    _persist_study(study_obj)
    logger.info(
        "[Trial %s] state=%s value=%s",
        trial_obj.number,
        trial_obj.state.name,
        trial_obj.value,
    )


In [ ]:
study.optimize(
    objective,
    n_trials=N_TRIALS,
    timeout=TIMEOUT_SECONDS,
    callbacks=[after_trial],
    gc_after_trial=True,
    show_progress_bar=False,
)
_persist_study(study)

print("Search complete")
print("Run directory:", RUN_DIR)
complete_trials = [trial for trial in study.trials if trial.value is not None]
if complete_trials:
    print("Best value:", study.best_value)
    print("Best params:", study.best_params)
else:
    print("No completed trials yet.")

trials_csv = RUN_DIR / "trials.csv"
if trials_csv.exists():
    trials_frame = pd.read_csv(trials_csv)
    if "value" in trials_frame.columns:
        display(trials_frame.sort_values("value", ascending=False).head(10))
    else:
        display(trials_frame.head(10))
else:
    print("No trials.csv written yet.")


In [ ]:
best_path = RUN_DIR / "best_hyperparameters.json"
if best_path.exists():
    best_payload = json.loads(best_path.read_text(encoding="utf-8"))
    print(json.dumps(best_payload, indent=2, sort_keys=True))
else:
    print("No completed trials yet.")
